# Week 3 - Synthetic Dataset Quality Auditor

This notebook compares a synthetic dataset CSV against a reference CSV and generates a markdown quality report.

Checks included:
- schema overlap
- missingness and uniqueness
- numeric drift (mean/std)
- categorical drift (TVD)

In [ ]:
from pathlib import Path
import importlib
import subprocess
import sys


def ensure_package(module_name: str, package_name: str | None = None):
    package_name = package_name or module_name
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])


ensure_package("pandas")

import pandas as pd

In [ ]:
def _safe_div(a: float, b: float) -> float:
    if b == 0:
        return 0.0
    return a / b


def _format_float(value: float) -> str:
    return f"{value:.4f}"


def _display_col(col: object) -> str:
    text = str(col)
    # Keep markdown tables stable even for unusual column names.
    text = text.replace("`", "\\`").replace("|", "\\|").replace("\n", " ")
    return text


def _distribution_distance(ref: pd.Series, syn: pd.Series) -> float:
    ref_dist = ref.value_counts(normalize=True, dropna=False)
    syn_dist = syn.value_counts(normalize=True, dropna=False)
    categories = ref_dist.index.union(syn_dist.index)
    ref_aligned = ref_dist.reindex(categories, fill_value=0.0)
    syn_aligned = syn_dist.reindex(categories, fill_value=0.0)
    return float(0.5 * (ref_aligned - syn_aligned).abs().sum())


def _numeric_metrics(ref: pd.Series, syn: pd.Series) -> dict[str, float]:
    ref_clean = ref.dropna()
    syn_clean = syn.dropna()
    ref_mean = float(ref_clean.mean()) if not ref_clean.empty else 0.0
    syn_mean = float(syn_clean.mean()) if not syn_clean.empty else 0.0
    ref_std = float(ref_clean.std(ddof=0)) if len(ref_clean) > 1 else 0.0
    syn_std = float(syn_clean.std(ddof=0)) if len(syn_clean) > 1 else 0.0
    mean_delta = abs(syn_mean - ref_mean)
    std_delta = abs(syn_std - ref_std)
    return {
        "ref_mean": ref_mean,
        "syn_mean": syn_mean,
        "mean_delta": mean_delta,
        "ref_std": ref_std,
        "syn_std": syn_std,
        "std_delta": std_delta,
    }


def _load_csv_pair(reference_path: Path, synthetic_path: Path) -> tuple[pd.DataFrame, pd.DataFrame, str]:
    # First try default header parsing.
    ref_default = pd.read_csv(reference_path)
    syn_default = pd.read_csv(synthetic_path)

    ref_cols = [str(c) for c in ref_default.columns]
    syn_cols = [str(c) for c in syn_default.columns]
    shared_cols = len(set(ref_cols).intersection(set(syn_cols)))

    def _looks_like_bad_header(col: str) -> bool:
        col_stripped = col.strip()
        return (
            "\n" in col_stripped
            or len(col_stripped) > 80
            or col_stripped.isdigit()
            or ":" in col_stripped
        )

    suspicious_header_count = sum(_looks_like_bad_header(c) for c in ref_cols + syn_cols)

    # Fallback to headerless if overlap is weak and header tokens look suspicious.
    if shared_cols == 0 or (shared_cols <= 1 and suspicious_header_count >= 2):
        ref_df = pd.read_csv(reference_path, header=None)
        syn_df = pd.read_csv(synthetic_path, header=None)

        max_cols = max(ref_df.shape[1], syn_df.shape[1])
        column_names = [f"col_{i}" for i in range(max_cols)]
        ref_df.columns = column_names[: ref_df.shape[1]]
        syn_df.columns = column_names[: syn_df.shape[1]]
        return ref_df, syn_df, "Loaded as headerless CSVs (assigned col_0, col_1, ...)."

    ref_default.columns = [str(c) for c in ref_default.columns]
    syn_default.columns = [str(c) for c in syn_default.columns]
    return ref_default, syn_default, "Loaded with CSV headers."


def build_report(reference_path: Path, synthetic_path: Path) -> str:
    ref_df, syn_df, load_mode = _load_csv_pair(reference_path, synthetic_path)

    ref_cols = set(ref_df.columns)
    syn_cols = set(syn_df.columns)
    common_cols = sorted(ref_cols.intersection(syn_cols))
    missing_in_synthetic = sorted(ref_cols - syn_cols)
    extra_in_synthetic = sorted(syn_cols - ref_cols)

    report: list[str] = []
    report.append("# Synthetic Dataset Quality Report")
    report.append("")
    report.append("## File Overview")
    report.append(f"- Parsing mode: `{load_mode}`")
    report.append(f"- Reference rows: `{len(ref_df)}`")
    report.append(f"- Synthetic rows: `{len(syn_df)}`")
    report.append(f"- Reference columns: `{len(ref_cols)}`")
    report.append(f"- Synthetic columns: `{len(syn_cols)}`")
    report.append(f"- Common columns: `{len(common_cols)}`")
    report.append("")

    report.append("## Schema Alignment")
    report.append(
        f"- Column overlap score: `{_format_float(_safe_div(len(common_cols), max(len(ref_cols), 1)))}`"
    )
    report.append(f"- Missing in synthetic: `{missing_in_synthetic or 'None'}`")
    report.append(f"- Extra in synthetic: `{extra_in_synthetic or 'None'}`")
    report.append("")

    report.append("## Missingness & Uniqueness")
    report.append("| Column | Ref missing % | Syn missing % | Ref unique % | Syn unique % |")
    report.append("|---|---:|---:|---:|---:|")
    for col in common_cols:
        ref_missing = _safe_div(ref_df[col].isna().sum(), len(ref_df)) * 100
        syn_missing = _safe_div(syn_df[col].isna().sum(), len(syn_df)) * 100
        ref_unique = _safe_div(ref_df[col].nunique(dropna=False), len(ref_df)) * 100
        syn_unique = _safe_div(syn_df[col].nunique(dropna=False), len(syn_df)) * 100
        col_label = _display_col(col)
        report.append(
            f"| `{col_label}` | {_format_float(ref_missing)} | {_format_float(syn_missing)} | "
            f"{_format_float(ref_unique)} | {_format_float(syn_unique)} |"
        )
    report.append("")

    numeric_cols = [
        c
        for c in common_cols
        if pd.api.types.is_numeric_dtype(ref_df[c]) and pd.api.types.is_numeric_dtype(syn_df[c])
    ]
    if numeric_cols:
        report.append("## Numeric Column Drift")
        report.append("| Column | Ref mean | Syn mean | |delta mean| | Ref std | Syn std | |delta std| |")
        report.append("|---|---:|---:|---:|---:|---:|---:|")
        for col in numeric_cols:
            metrics = _numeric_metrics(ref_df[col], syn_df[col])
            col_label = _display_col(col)
            report.append(
                f"| `{col_label}` | {_format_float(metrics['ref_mean'])} | {_format_float(metrics['syn_mean'])} | "
                f"{_format_float(metrics['mean_delta'])} | {_format_float(metrics['ref_std'])} | "
                f"{_format_float(metrics['syn_std'])} | {_format_float(metrics['std_delta'])} |"
            )
        report.append("")

    categorical_cols = [c for c in common_cols if c not in numeric_cols]
    if categorical_cols:
        report.append("## Categorical Distribution Distance (TVD)")
        report.append("| Column | TVD distance |")
        report.append("|---|---:|")
        for col in categorical_cols:
            tvd = _distribution_distance(ref_df[col], syn_df[col])
            col_label = _display_col(col)
            report.append(f"| `{col_label}` | {_format_float(float(tvd))} |")
        report.append("")

    report.append("## Quick Interpretation")
    report.append("- Lower drift and lower TVD generally indicate better alignment.")
    report.append("- Some drift is normal when synthetic row count differs from reference.")
    report.append("- Use this report as a first-pass quality check, not a privacy guarantee.")
    report.append("")

    return "\n".join(report)

In [ ]:
# Configuration block (edit only this block for new datasets)
data_dir = Path(".")
output_report = Path("quality_report.md")

# Optional manual override. Set both to None to use auto-detection.
manual_reference_csv = Path("../../../../week6/human_in.csv")
manual_synthetic_csv = Path("../../../../week6/human_out.csv")

REFERENCE_NAME_HINTS = ("reference", "real", "original", "ground_truth", "baseline", "_in", "input")
SYNTHETIC_NAME_HINTS = ("synthetic", "generated", "fake", "sampled", "output", "_out", "train", "test", "valid")
IGNORE_DIR_PARTS = {".git", ".venv", "venv", "site-packages", "node_modules", "__pycache__"}


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / ".git").exists():
            return candidate
    return current


def is_ignored_path(csv_path: Path) -> bool:
    return any(part.lower() in IGNORE_DIR_PARTS for part in csv_path.parts)


def has_hint(path_obj: Path, hints: tuple[str, ...]) -> bool:
    name = path_obj.name.lower()
    return any(hint in name for hint in hints)


def base_reference_name(csv_path: Path) -> str:
    stem = csv_path.stem.lower()
    for token in ("_reference", "-reference", " reference", "_real", "-real", " real"):
        stem = stem.replace(token, "")
    return stem.strip("_-")


def pick_synthetic_for_reference(reference_csv: Path, candidates: list[Path]) -> Path | None:
    local_candidates = [p for p in candidates if p != reference_csv and p.parent == reference_csv.parent]

    for csv_path in local_candidates:
        if has_hint(csv_path, SYNTHETIC_NAME_HINTS):
            return csv_path

    ref_base = base_reference_name(reference_csv)
    if ref_base:
        for csv_path in local_candidates:
            if ref_base in csv_path.stem.lower():
                return csv_path

    return None


def resolve_input_path(path_value: Path | str | None, project_root: Path) -> Path | None:
    if not path_value:
        return None

    candidate = Path(path_value)
    if candidate.is_absolute():
        return candidate

    from_cwd = (Path.cwd() / candidate).resolve()
    if from_cwd.exists():
        return from_cwd

    from_project = (project_root / candidate).resolve()
    if from_project.exists():
        return from_project

    return from_cwd


def auto_detect_inputs(data_dir: Path, project_root: Path) -> tuple[Path | None, Path | None, list[Path], Path]:
    search_root = data_dir.resolve() if data_dir.exists() else project_root
    csv_candidates = sorted([p for p in search_root.rglob("*.csv") if not is_ignored_path(p)])

    if not csv_candidates and search_root != project_root:
        search_root = project_root
        csv_candidates = sorted([p for p in search_root.rglob("*.csv") if not is_ignored_path(p)])

    reference_candidates = [p for p in csv_candidates if has_hint(p, REFERENCE_NAME_HINTS)]
    auto_reference_csv = reference_candidates[0] if reference_candidates else None
    auto_synthetic_csv = pick_synthetic_for_reference(auto_reference_csv, csv_candidates) if auto_reference_csv else None

    return auto_reference_csv, auto_synthetic_csv, csv_candidates, search_root

In [ ]:
# Execution block
project_root = find_project_root(Path.cwd())
manual_ref = resolve_input_path(manual_reference_csv, project_root)
manual_syn = resolve_input_path(manual_synthetic_csv, project_root)

auto_ref, auto_syn, csv_candidates, search_root = auto_detect_inputs(data_dir, project_root)
reference_csv = manual_ref or auto_ref
synthetic_csv = manual_syn or auto_syn

if reference_csv and synthetic_csv:
    if manual_ref and manual_syn:
        print(f"Using manual reference CSV: {reference_csv.as_posix()}")
        print(f"Using manual synthetic CSV: {synthetic_csv.as_posix()}")
    else:
        print(f"Auto-detected reference CSV: {reference_csv.as_posix()}")
        print(f"Auto-detected synthetic CSV: {synthetic_csv.as_posix()}")
        print(f"Search root: {search_root.as_posix()}")

    report = build_report(reference_csv, synthetic_csv)
    output_report.parent.mkdir(parents=True, exist_ok=True)
    output_report.write_text(report, encoding="utf-8")
    print(f"Quality report written to: {output_report.resolve()}")
    print("\n".join(report.splitlines()[:30]))
else:
    print("Could not auto-detect both reference and synthetic CSV files safely.")
    print(f"Searched folder: {search_root.as_posix()}")
    print(f"CSV files found (after filtering env folders): {len(csv_candidates)}")

    if auto_ref and not auto_syn:
        print(f"Reference candidate found: {auto_ref.as_posix()}")
        print("No synthetic candidate found in the same folder as the reference file.")

    if csv_candidates:
        print("Candidates:")
        for csv_path in csv_candidates[:20]:
            print(f"- {csv_path.as_posix()}")
        if len(csv_candidates) > 20:
            print(f"... and {len(csv_candidates) - 20} more")

    print("Tip: set data_dir to a folder that contains both your CSV files, or set manual_reference_csv/manual_synthetic_csv.")